# 03 — ML Inference QA

Quality-assurance notebook for the two machine-learning inference layers:

1. **Pole detection** (`PoleDetector`) — Stanford GridMapping CNN (Wang et al. 2023)
   Compares OSM-mapped poles vs ML-inferred poles from street-view imagery.

2. **Gridfinder rural lines** (`GridfinderRunner`) — MV distribution line prediction
   (Arderne et al. 2020) for rural Ontario zones where OSM has no geometry.

Confidence levels: OSM confirmed = 0.90 · ML urban = 0.65 · gridfinder rural = 0.30

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import box

from src.ingestion.osm_fetcher import OSMFetcher
from src.ml_inference.pole_detector import PoleDetector
from src.ml_inference.gridfinder_runner import GridfinderRunner
from src.ml_inference.transformer_locator import TransformerLocator
from src.utils.config_loader import load_settings

cfg = load_settings()
bbox_cfg = cfg["region"]["bbox"]
BBOX = (bbox_cfg["west"], bbox_cfg["south"], bbox_cfg["east"], bbox_cfg["north"])

print("ML Inference QA — Mississauga pilot bbox")
print(f"  bbox: {BBOX}")

## 3.1  Load OSM Pole Baseline

In [ ]:
osm = OSMFetcher()
osm_power = osm.fetch_power_features(bbox=BBOX)

# OSM pole and tower points
osm_poles = osm_power[
    osm_power["power"].isin(["pole", "tower"]) &
    (osm_power.geometry.geom_type == "Point")
].copy()
osm_poles["source"] = "OSM"
osm_poles["confidence"] = 0.90

print(f"OSM poles in bbox:   {len(osm_poles):,}")
print(f"  power=pole:        {(osm_poles['power'] == 'pole').sum():,}")
print(f"  power=tower:       {(osm_poles['power'] == 'tower').sum():,}")

## 3.2  Load ML-Inferred Poles

In [ ]:
# Load pre-computed ML pole detections (from PoleDetector output parquet)
ML_POLES_PATH = Path(cfg["outputs"]["ml_poles_parquet"])

if ML_POLES_PATH.exists():
    ml_poles = gpd.read_parquet(ML_POLES_PATH)
    # Filter to pilot bbox
    bbox_poly = box(*BBOX)
    ml_poles = ml_poles[ml_poles.within(bbox_poly)].copy()
    print(f"ML-inferred poles loaded: {len(ml_poles):,}")
else:
    print(f"ML poles parquet not found at {ML_POLES_PATH}")
    print("Generating stub data for QA demo...")
    # Stub: simulate ML pole detections with realistic confidence distribution
    rng = np.random.default_rng(seed=42)
    n_ml = 840
    lons = rng.uniform(BBOX[0], BBOX[2], n_ml)
    lats = rng.uniform(BBOX[1], BBOX[3], n_ml)
    # ML confidences: bimodal — high-confidence urban detections + lower suburban
    conf_urban = rng.beta(8, 2, int(n_ml * 0.6))  # mean ~0.80
    conf_suburban = rng.beta(3, 5, n_ml - int(n_ml * 0.6))  # mean ~0.38
    confidences = np.concatenate([conf_urban, conf_suburban])
    from shapely.geometry import Point
    ml_poles = gpd.GeoDataFrame(
        {
            "confidence": confidences,
            "source": "GridMapping_ML",
            "node_type": "pole",
            "inferred": True,
            "geometry": [Point(lo, la) for lo, la in zip(lons, lats)],
        },
        crs="EPSG:4326",
    )
    print(f"Stub ML poles generated: {len(ml_poles):,}")

## 3.3  OSM vs ML Coverage Comparison

In [ ]:
# Spatial density: divide bbox into a 10x10 grid and compare counts per cell
WEST, SOUTH, EAST, NORTH = BBOX
nx_cells, ny_cells = 10, 10
lon_edges = np.linspace(WEST, EAST, nx_cells + 1)
lat_edges = np.linspace(SOUTH, NORTH, ny_cells + 1)

def bin_points(gdf, lon_edges, lat_edges):
    """Return 2D array of point counts per grid cell."""
    lons = gdf.geometry.x.values
    lats = gdf.geometry.y.values
    counts, _, _ = np.histogram2d(lons, lats, bins=[lon_edges, lat_edges])
    return counts.T  # transpose: rows=lat, cols=lon

osm_grid = bin_points(osm_poles, lon_edges, lat_edges)
ml_grid  = bin_points(ml_poles, lon_edges, lat_edges)
ratio_grid = np.where(osm_grid > 0, ml_grid / osm_grid, np.nan)

print(f"OSM poles in bbox:  {int(osm_grid.sum()):,}")
print(f"ML poles in bbox:   {int(ml_grid.sum()):,}")
print(f"Overall ML/OSM ratio: {ml_grid.sum() / max(osm_grid.sum(), 1):.2f}")
print(f"Cells where ML > OSM: {(ml_grid > osm_grid).sum()} / {nx_cells * ny_cells}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Pole Coverage: OSM vs ML-Inferred (Mississauga bbox)", fontsize=13, fontweight="bold")

extent = [WEST, EAST, SOUTH, NORTH]

im0 = axes[0].imshow(osm_grid, origin="lower", extent=extent, cmap="Blues", aspect="auto")
axes[0].set_title(f"OSM Poles\n({int(osm_grid.sum()):,} total)")
plt.colorbar(im0, ax=axes[0], label="count per cell")

im1 = axes[1].imshow(ml_grid, origin="lower", extent=extent, cmap="Greens", aspect="auto")
axes[1].set_title(f"ML-Inferred Poles\n({int(ml_grid.sum()):,} total)")
plt.colorbar(im1, ax=axes[1], label="count per cell")

im2 = axes[2].imshow(ratio_grid, origin="lower", extent=extent, cmap="RdYlGn",
                      vmin=0, vmax=3, aspect="auto")
axes[2].set_title("ML / OSM Ratio\n(green > 1x coverage, red < 1x)")
plt.colorbar(im2, ax=axes[2], label="ML / OSM ratio")

for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig("../data/outputs/03_pole_coverage_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 3.4  Confidence Distribution by Source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Pole Detection — Confidence Score Distribution by Source", fontsize=12, fontweight="bold")

# OSM confidence (fixed at 0.90)
ax = axes[0]
osm_conf = np.full(len(osm_poles), 0.90)
ax.hist(osm_conf, bins=20, range=(0, 1), color="#2980b9", edgecolor="white", alpha=0.85)
ax.axvline(0.90, color="#c0392b", linestyle="--", linewidth=1.5, label="OSM confirmed = 0.90")
ax.set_title(f"OSM Poles (n={len(osm_poles):,})")
ax.set_xlabel("Confidence")
ax.set_ylabel("Count")
ax.set_xlim(0, 1)
ax.legend(fontsize=8)

# ML confidence (variable)
ax = axes[1]
ml_conf = ml_poles["confidence"].values
above_threshold = (ml_conf >= cfg["ml_inference"]["pole_detector"]["confidence_threshold"]).sum()
thresh = cfg["ml_inference"]["pole_detector"]["confidence_threshold"]
ax.hist(ml_conf, bins=30, range=(0, 1), color="#27ae60", edgecolor="white", alpha=0.85)
ax.axvline(thresh, color="#e74c3c", linestyle="--", linewidth=1.5,
           label=f"Threshold = {thresh:.2f}")
ax.axvline(0.65, color="#f39c12", linestyle=":", linewidth=1.5,
           label="ML urban nominal = 0.65")
ax.set_title(f"ML-Inferred Poles (n={len(ml_poles):,})\n{above_threshold:,} above threshold")
ax.set_xlabel("Confidence")
ax.set_ylabel("Count")
ax.set_xlim(0, 1)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../data/outputs/03_confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ML median confidence: {np.median(ml_conf):.3f}")
print(f"ML mean confidence:   {np.mean(ml_conf):.3f}")
print(f"ML poles above threshold ({thresh:.2f}): {above_threshold:,} / {len(ml_poles):,}")

## 3.5  Gridfinder Rural Predictions vs OSM Lines

In [ ]:
# Load gridfinder predicted lines for a rural Ontario area outside Mississauga
RURAL_BBOX = (-80.10, 43.50, -79.60, 43.80)  # rural Halton/Wellington fringe

GF_OUTPUT_PATH = Path(cfg["outputs"]["gridfinder_lines_geojson"])

if GF_OUTPUT_PATH.exists():
    gf_lines = gpd.read_file(GF_OUTPUT_PATH)
    rural_bbox_poly = box(*RURAL_BBOX)
    gf_lines = gf_lines[gf_lines.intersects(rural_bbox_poly)].copy()
    print(f"Gridfinder lines loaded: {len(gf_lines):,}")
else:
    print(f"Gridfinder output not found at {GF_OUTPUT_PATH}")
    print("Showing GridfinderRunner configuration only.")
    gf_runner = GridfinderRunner()
    print(f"  prediction_threshold: {gf_runner._threshold}")
    gf_lines = None

# OSM lines in the same rural area
osm_rural = OSMFetcher()
osm_rural_power = osm_rural.fetch_power_features(bbox=RURAL_BBOX)
osm_rural_lines = osm_rural_power[
    osm_rural_power["power"].isin(["line", "minor_line", "cable"])
].copy()
print(f"OSM lines in rural bbox: {len(osm_rural_lines):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Gridfinder Rural MV Predictions vs OSM Lines\n"
    f"Rural bbox: {RURAL_BBOX}",
    fontsize=12, fontweight="bold"
)

RW, RS, RE, RN = RURAL_BBOX

# OSM panel
ax = axes[0]
if len(osm_rural_lines) > 0:
    osm_rural_lines.plot(ax=ax, color="#2980b9", linewidth=1.2, alpha=0.9)
ax.set_title(f"OSM Lines ({len(osm_rural_lines):,} features)\nconf=0.90 (OSM confirmed)")
ax.set_xlim(RW, RE)
ax.set_ylim(RS, RN)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_facecolor("#f8f9fa")

# Gridfinder panel
ax = axes[1]
if gf_lines is not None and len(gf_lines) > 0:
    gf_lines.plot(ax=ax, color="#e67e22", linewidth=1.0, alpha=0.75)
    n_gf = len(gf_lines)
    title_text = f"Gridfinder Predictions ({n_gf:,} features)\nconf=0.30 (gridfinder_rural)"
else:
    title_text = "Gridfinder (output not generated)\nconf=0.30 (gridfinder_rural)"
ax.set_title(title_text)
ax.set_xlim(RW, RE)
ax.set_ylim(RS, RN)
ax.set_xlabel("Longitude")
ax.set_facecolor("#f8f9fa")

# Add confidence legend
legend_handles = [
    mpatches.Patch(color="#2980b9", label="OSM confirmed (0.90)"),
    mpatches.Patch(color="#e67e22", label="gridfinder rural (0.30)"),
]
axes[1].legend(handles=legend_handles, fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig("../data/outputs/03_gridfinder_vs_osm.png", dpi=150, bbox_inches="tight")
plt.show()

## 3.6  ML Inference Summary

In [ ]:
print("=" * 55)
print("  ML INFERENCE QA SUMMARY")
print("=" * 55)
print()
print("POLE DETECTION (GridMapping CNN)")
print(f"  OSM poles in pilot bbox:       {len(osm_poles):>7,}  conf=0.90")
print(f"  ML-inferred poles:             {len(ml_poles):>7,}  conf=variable")
print(f"  ML poles above threshold:      {above_threshold:>7,}")
print(f"  ML median confidence:          {np.median(ml_conf):>10.3f}")
print(f"  Coverage ratio (ML/OSM):       {len(ml_poles)/max(len(osm_poles),1):>10.2f}x")
print()
print("GRIDFINDER RURAL MV LINES")
print(f"  OSM lines in rural bbox:       {len(osm_rural_lines):>7,}  conf=0.90")
if gf_lines is not None:
    print(f"  Gridfinder predicted lines:    {len(gf_lines):>7,}  conf=0.30")
else:
    print(f"  Gridfinder predicted lines:    {'N/A':>7}  (output not generated)")
print("=" * 55)